# Meal-Demand Forecasting — Demonstration Notebook

**Offline-First Data Synchronization System for School Feeding Programs**

This notebook demonstrates the forecasting pipeline from the project. It does **not** reimplement anything:
every step imports the project's own code from `forecasting/` (`models/`, `services/`, `config/`), so it uses the
same target, preprocessing, model settings and evaluation method as the Flask service.

> **Synthetic data.** The dataset represents **one simulated Kenyan school** generated under documented
> assumptions (`docs/synthetic-dataset.md`). Results compare forecasting methods on simulated data; they are
> **not** evidence of accuracy for any real Kenyan school.

**How to run:** run the cells from top to bottom. Steps 0.1–0.2 are only needed once per Colab session.

## 0.1 Get the project files

Choose how the files reach Colab:

* **upload** (simplest): you will be asked to pick `school_feeding_forecasting_colab.zip` from your computer.
* **drive**: put the zip in Google Drive and set `DRIVE_ZIP_PATH`.
* **github**: clones the repository (only works once the Phase 9–10 work has been pushed).

In [ ]:
# @title 0.1 Get the project files
import pathlib, subprocess, zipfile

SOURCE = "upload"  # @param ["upload", "drive", "github"]
DRIVE_ZIP_PATH = "/content/drive/MyDrive/school_feeding_forecasting_colab.zip"  # @param {type:"string"}
GITHUB_URL = "https://github.com/Gichezz/School-Feeding-Program-System.git"  # @param {type:"string"}

if pathlib.Path("/content/forecasting/models/data.py").exists() or \
   pathlib.Path("/content/School-Feeding-Program-System/forecasting/models/data.py").exists():
    print("Project files already present - nothing to do.")
elif SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()                      # pick school_feeding_forecasting_colab.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    zipfile.ZipFile(DRIVE_ZIP_PATH).extractall("/content")
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, "/content/School-Feeding-Program-System"], check=True)
print("Done.")

## 0.2 Install the pinned libraries

These are the same versions as `forecasting/requirements.txt`. The saved Prophet model was created with
Prophet 1.1.6, and Prophet 1.1.6 needs cmdstanpy 1.2.x. pip may print dependency-conflict warnings about
other Colab packages; they do not affect this notebook.

**After this cell finishes, restart the session (Runtime → Restart session)** so the pinned
`numpy`/`pandas` versions are loaded, then continue from step 0.3. You do not need to rerun 0.1 or 0.2.

In [ ]:
# @title 0.2 Install pinned libraries (then restart the session)
!pip install -q "prophet==1.1.6" "cmdstanpy==1.2.5" "pandas==2.2.3" "numpy==2.1.3" "Flask==3.0.3" "PyYAML>=6.0" pytest

## 0.3 Load the project code

In [ ]:
import logging, os, sys, warnings
from importlib.metadata import version
from pathlib import Path

CANDIDATES = [Path("/content/forecasting"), Path("/content/School-Feeding-Program-System/forecasting"),
              Path.cwd() / "forecasting", Path.cwd(), Path.cwd().parent]
FORECASTING_DIR = next((p.resolve() for p in CANDIDATES if (p / "models" / "data.py").exists()), None)
if FORECASTING_DIR is None:
    raise FileNotFoundError("Project files not found. Run step 0.1 first.")
for p in (FORECASTING_DIR, FORECASTING_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
os.chdir(FORECASTING_DIR)

EXPECTED = {"prophet": "1.1.6", "cmdstanpy": "1.2.5", "pandas": "2.2.3", "numpy": "2.1.3"}
found = {k: version(k) for k in EXPECTED}
print("Project folder:", FORECASTING_DIR)
print("Library versions:", found)
if found != EXPECTED:
    print("WARNING: versions differ from the project. In Colab: run step 0.2, then Runtime -> Restart session.")

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Markdown

# Project code (the same modules the Flask service and training scripts use)
from models.data import load_forecast_config, load_dataset, preprocess, split_by_dates, resolve
from models.features import add_calendar_features, future_school_days
from models.baselines import HistoricalAverage, MovingAverage
from models.evaluation import forecast_windows, run_baseline, summarise, summary_records, mean_metric
from models.prophet_model import training_frame, fit_prophet, predict_prophet, load_model
from models.features import last_known_enrollment
from services.forecast_service import ForecastService, ForecastError
import evaluate_forecasting_models as evaluation_script

logging.getLogger("cmdstanpy").disabled = True  # hide per-fit Stan progress lines (errors still raise)

cfg = load_forecast_config()
TARGET = cfg["dataset"]["target"]
print("Target:", TARGET)

### Chart style

One shared style for every chart. Colours follow the model, never its rank: **Prophet = blue**,
**historical average = orange**, **moving average = aqua**, and actual demand is dark ink. Every chart has a legend,
and the values behind each chart are also shown as a table.

In [ ]:
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"
COLORS = {"prophet": "#2a78d6", "historical_average": "#eb6834", "moving_average": "#1baf7a"}
BAND = "#d6d5cf"

def model_color(name):
    return next((c for k, c in COLORS.items() if name.startswith(k)), INK_2)

def pretty(name):
    return (name.replace("prophet_calendar_enrollment__cps_0.01", "Prophet (selected)")
                .replace("historical_average_trailing_365d", "Historical avg (365 d)")
                .replace("historical_average_expanding", "Historical avg (expanding)")
                .replace("moving_average_", "Moving avg ").replace("prophet_", "Prophet "))

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "legend.frameon": False, "font.size": 10.5, "lines.linewidth": 2,
    "figure.dpi": 110,
})

def legend_above(ax, ncols):
    """Legend in a row above the plot area, so it never covers data."""
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncols=ncols, borderaxespad=0, handlelength=1.8)

## 1. Dataset loading and exploration

`load_dataset` reads the CSV, parses `date`, parses boolean columns and sorts chronologically.

In [ ]:
dataset_path = resolve(cfg["dataset"]["path"])
raw = load_dataset(dataset_path)
print(f"{dataset_path.name}: {len(raw):,} rows x {raw.shape[1]} columns, "
      f"{raw['date'].min().date()} to {raw['date'].max().date()}")
print(f"School days: {raw['is_school_day'].sum():,}   Non-school days: {(~raw['is_school_day']).sum():,}")
print("Source type:", raw["source_type"].unique(), "| Scenario:", raw["synthetic_scenario"].unique())
display(raw.head(10))

In [ ]:
display(Markdown("**Target on school days vs non-school days** (missing on non-school days by design: no session, no meals)"))
display(pd.DataFrame({
    "school days": raw.loc[raw["is_school_day"], TARGET].describe(),
    "non-school days": raw.loc[~raw["is_school_day"], TARGET].describe(),
}).round(1))

In [ ]:
school_raw = raw[raw["is_school_day"]]
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(school_raw["date"], school_raw[TARGET], color=INK, linewidth=0.6, label="Estimated meal demand")
axes[0].set_title("Estimated meal demand per school day (1998-2025)")
axes[0].set_ylabel("meals")
axes[1].plot(school_raw["date"], school_raw["students_enrolled"], color=INK_2, linewidth=1.2, label="Students enrolled")
axes[1].set_title("Students enrolled (separate chart - different measure)")
axes[1].set_ylabel("students")
for ax in axes:
    ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
one_year = raw[raw["date"].dt.year == 2024].set_index("date")
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.plot(one_year.index, one_year[TARGET], color=INK, linewidth=1.2, label="Demand (gaps = no school)")
for term, g in one_year[one_year["term"] != "holiday"].groupby("term"):
    ax.axvspan(g.index.min(), g.index.max(), color=BAND, alpha=0.35, linewidth=0)
    ax.text(g.index.min(), ax.get_ylim()[1], f" {term}", va="top", color=INK_2, fontsize=9)
ax.set_title("One year in detail (2024): demand exists only on school days; shaded = term")
ax.set_ylabel("meals"); ax.legend(loc="lower right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
plt.tight_layout(); plt.show()

## 2. Preprocessing

`preprocess` checks for duplicate dates, gaps, missing and negative targets, then keeps **school days only**.
Non-school days are dropped, **not** set to zero. `add_calendar_features` adds the calendar regressors that are
known in advance (first day of term, first week of term, exam period).

In [ ]:
school, summary = preprocess(raw, TARGET, cfg["dataset"]["date_column"], cfg["dataset"]["school_day_column"])
school = add_calendar_features(school)
display(pd.Series(summary.as_dict(), name="value").to_frame())
display(school[["date", "day_of_week", "term", "week_of_term", "students_enrolled",
                "first_day_of_term", "first_week_of_term", "exam_period", TARGET]].head(8))

## 3. Chronological split

Train → validation → test, never shuffled. Validation is used to choose and tune models; the test period is used
once, for the final comparison.

In [ ]:
splits = split_by_dates(school, cfg["split"])
display(pd.DataFrame([{"split": n, "configured": f"{cfg['split'][n]['start']} .. {cfg['split'][n]['end']}",
                       "first school day": f["date"].min().date(), "last school day": f["date"].max().date(),
                       "school days": len(f)} for n, f in splits.items()]))
assert splits["train"]["date"].max() < splits["validation"]["date"].min() < splits["test"]["date"].min()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.plot(school["date"], school[TARGET], color=INK, linewidth=0.5, label="Estimated meal demand")
top = ax.get_ylim()[1]
for name in ("train", "validation", "test"):
    s, e = pd.Timestamp(cfg["split"][name]["start"]), pd.Timestamp(cfg["split"][name]["end"])
    if name != "train":
        ax.axvspan(s, e, color=BAND, alpha=0.6 if name == "test" else 0.35, linewidth=0)
    ax.text(s + (e - s) / 2, top, "val" if name == "validation" else name, ha="center", va="bottom", color=INK_2)
ax.set_title("Chronological split: train 1998-2021, validation 2022-2023, test 2024-2025", pad=22)
ax.set_ylabel("meals"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

### Rolling forecast origins

A **forecast origin** is the day a forecast is made, using only data up to the previous day. Origins are placed
every 14 days through the validation (and later the test) period. From each origin, every model forecasts every
school day in the next 30 calendar days; horizons of 7, 14 and 30 days score the targets that far ahead.

In [ ]:
ev = cfg["evaluation"]
STEP, HORIZONS, MAX_H = ev["origin_step_days"], ev["horizons_days"], max(ev["horizons_days"])
val_windows = forecast_windows(school, cfg["split"]["validation"]["start"], cfg["split"]["validation"]["end"], STEP, MAX_H)
test_windows = forecast_windows(school, cfg["split"]["test"]["start"], cfg["split"]["test"]["end"], STEP, MAX_H)
print(f"Validation origins: {len(val_windows)} | Test origins: {len(test_windows)}")

origin, history, targets = test_windows[20]
print(f"Example origin {origin.date()}: history = {len(history):,} school days up to {history['date'].max().date()}; "
      f"targets = {len(targets)} school days {targets['date'].min().date()}..{targets['date'].max().date()}")
assert history["date"].max() < origin <= targets["date"].min()   # no future data in history

## 4. Historical-average baseline

One level from school days **before** the origin, used for every day of the horizon:
`expanding` = mean of all history; `trailing_365d` = mean of the previous 365 calendar days.
These run live in the notebook (they take seconds).

In [ ]:
ha_models = [HistoricalAverage(v.get("window_days")) for v in cfg["baselines"]["historical_average"]["variants"]]
ha_val = pd.concat([run_baseline(m, val_windows, TARGET) for m in ha_models], ignore_index=True)
ha_summary = summarise(ha_val, HORIZONS, ev["mape_min_actual"])
display(ha_summary[["model", "horizon_days", "n", "mae", "rmse", "wape", "bias"]].round(3))

## 5. Moving-average baseline

The mean of the last *k* school days before the origin (k = 5, 10, 20), held constant across the horizon.
It never uses actual values from inside the horizon, so later forecast days do not peek at earlier ones.

In [ ]:
ma_models = [MovingAverage(w) for w in cfg["baselines"]["moving_average"]["windows_school_days"]]
ma_val = pd.concat([run_baseline(m, val_windows, TARGET) for m in ma_models], ignore_index=True)
ma_summary = summarise(ma_val, HORIZONS, ev["mape_min_actual"])
display(ma_summary[["model", "horizon_days", "n", "mae", "rmse", "wape", "bias"]].round(3))

# Validation selection (lowest mean MAE across horizons) - same rule as the project script
sel_ha = min((m.name for m in ha_models), key=lambda n: mean_metric(ha_summary, n, "mae"))
sel_ma = min((m.name for m in ma_models), key=lambda n: mean_metric(ma_summary, n, "mae"))
print("Selected on validation:", sel_ha, "and", sel_ma)

In [ ]:
# Test evaluation of the selected baselines, and a check that they match the project's saved results
base_test = pd.concat([run_baseline(next(m for m in ha_models + ma_models if m.name == n), test_windows, TARGET)
                       for n in (sel_ha, sel_ma)], ignore_index=True)
saved_preds = pd.read_csv(resolve(cfg["artifacts"]["evaluation_predictions"]), parse_dates=["origin", "date"])
saved_base = saved_preds[(saved_preds["split"] == "test") & saved_preds["model"].isin([sel_ha, sel_ma])]
merged = base_test.merge(saved_base, on=["model", "origin", "date"], suffixes=("", "_saved"))
assert len(merged) == len(base_test) == len(saved_base)
print("Max difference vs saved project results:", float((merged["yhat"] - merged["yhat_saved"]).abs().max()))

## 6. Prophet

Prophet is trained on school days only (`ds` = date, `y` = demand). Candidate configurations come from
`config/forecasting.yaml`: stage 1 compares regressor sets, stage 2 tunes the best set. Regressors are limited
to values that can be supplied for future dates: calendar flags from the school calendar, and enrollment
(the last known value, held constant). Rainfall, attendance and meals are never used.

In [ ]:
stage1 = evaluation_script.prophet_specs(cfg)
display(pd.DataFrame([{"name": s["name"], "regressors": ", ".join(s["regressors"]) or "(none)",
                       **{k: v for k, v in s["params"].items() if k in ("changepoint_prior_scale", "seasonality_mode")}}
                      for s in stage1]))
display(pd.DataFrame(cfg["prophet"]["tuning_grid"]).assign(applies_to="best stage-1 set"))

### 6.1 Train Prophet at one forecast origin (live)

This fits the selected configuration exactly as the evaluation does at one origin: history before the origin only,
enrollment held at its last known value.

In [ ]:
evaluation = json.loads(resolve(cfg["artifacts"]["evaluation_json"]).read_text())
spec = evaluation["selected"]["prophet"]
print("Selected Prophet configuration:", spec["name"])
print("  regressors:", spec["regressors"])
print("  params:", spec["params"])

origin, history, targets = test_windows[20]
pc = cfg["prophet"]
m = fit_prophet(training_frame(history, TARGET, spec["regressors"]), spec["params"], spec["regressors"],
                pc["interval_width"], pc["uncertainty_samples"])
future = pd.DataFrame({"ds": targets["date"].values})
for r in spec["regressors"]:
    future[r] = last_known_enrollment(history, origin) if r == "students_enrolled" else targets[r].astype(float).values
fc = predict_prophet(m, future, seed=pc["random_seed"])
one = targets[["date", TARGET]].reset_index(drop=True).join(fc[["yhat", "yhat_lower", "yhat_upper"]])
display(one.round(1).head(10))

saved_one = saved_preds[(saved_preds["model"] == spec["name"]) & (saved_preds["origin"] == origin)].sort_values("date")
print("Max difference vs saved project results at this origin:",
      round(float(np.abs(one["yhat"].values - saved_one["yhat"].values).max()), 4))

In [ ]:
fig = m.plot_components(m.predict(future.assign(ds=pd.to_datetime(future["ds"]))))
fig.suptitle("Prophet components at one origin (trend, weekly, yearly, regressors)", x=0.01, ha="left", fontweight="bold")
for ax in fig.axes:
    ax.ticklabel_format(axis="y", useOffset=False, style="plain")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout(); plt.show()

**Reading the components.** *Trend* is the slow level over the forecast window. *Weekly* shows Monday-Friday
differences. The Saturday/Sunday values are an extrapolation: there is no weekend data and the model is never asked
for weekends. *Yearly* is the seasonal shape; the dip in November-December falls in the long holiday, where
there is no data and no forecasts are made. *Extra regressors* are the calendar and enrollment effects
(here the jump is the exam period).

### 6.2 Full rolling-origin evaluation

The full evaluation refits Prophet at every origin for 6 configurations (about 400 fits). On a laptop it takes
about 7 minutes; Colab may be slower. By default this notebook **loads the project's saved results**, which were
produced by exactly this code. Set `RUN_FULL_EVALUATION = True` to recompute them here with the same settings
(it runs the project's own `scripts/evaluate_forecasting_models.py`).

In [ ]:
# @title 6.2 Load (or recompute) the full evaluation
RUN_FULL_EVALUATION = False  # @param {type:"boolean"}

if RUN_FULL_EVALUATION:
    import subprocess
    subprocess.run([sys.executable, "scripts/evaluate_forecasting_models.py"], check=True)

evaluation = json.loads(resolve(cfg["artifacts"]["evaluation_json"]).read_text())
preds = pd.read_csv(resolve(cfg["artifacts"]["evaluation_predictions"]), parse_dates=["origin", "date"])
val_summary = pd.DataFrame(evaluation["validation"])
test_summary = pd.DataFrame(evaluation["test"])
print("Evaluation generated at:", evaluation["generated_at_utc"], "| design:", evaluation["evaluation_design"])

prophet_val = val_summary[val_summary["model"].str.startswith("prophet")]
table = prophet_val.pivot(index="model", columns="horizon_days", values="mae").round(2)
table["mean MAE"] = table.mean(axis=1).round(3)
display(Markdown("**Validation MAE of every Prophet candidate** (lowest mean MAE is selected)"))
display(table.sort_values("mean MAE"))
print("Selected:", evaluation["selected"]["prophet"]["name"])

## 7. Model comparison

The three selected models on validation (used for selection) and on the test period (used once).
**MAE** = average absolute error in meals; **WAPE** = total absolute error ÷ total demand.

In [ ]:
selected = [evaluation["selected"]["historical_average"], evaluation["selected"]["moving_average"],
            evaluation["selected"]["prophet"]["name"]]
v = val_summary.set_index(["model", "horizon_days"])
rows = []
for _, t in test_summary.iterrows():
    vv = v.loc[(t["model"], t["horizon_days"])]
    rows.append({"model": pretty(t["model"]), "horizon (days)": t["horizon_days"],
                 "val MAE": vv["mae"], "val RMSE": vv["rmse"], "val WAPE %": 100 * vv["wape"], "val n": vv["n"],
                 "test MAE": t["mae"], "test RMSE": t["rmse"], "test WAPE %": 100 * t["wape"], "test n": t["n"]})
comparison = pd.DataFrame(rows).sort_values(["horizon (days)", "test MAE"])
display(comparison.round(2).reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.26
for i, name in enumerate(selected):
    s = test_summary[test_summary["model"] == name].sort_values("horizon_days")
    x = np.arange(len(s)) + (i - 1) * width
    ax.bar(x, s["mae"], width=width - 0.02, color=model_color(name), label=pretty(name))
ax.set_xticks(np.arange(len(HORIZONS)), [f"{h}-day horizon" for h in HORIZONS])
ax.set_ylabel("test MAE (meals)"); ax.set_ylim(0, None)
ax.set_title("Test-period error by horizon (lower is better)", pad=30)
legend_above(ax, 3)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Reading the results.** On the test years Prophet is off by about 12 meals per school day on average
(about 2.3%), and the best baseline by about 13–14 meals. The difference is real but small. Most of Prophet's
advantage comes from the enrollment regressor; without it Prophet was no better than the moving average on
validation (see the candidate table in 6.2). Simple baselines are strong here because the simulated demand is
stable from week to week.

Note: forecasts from neighbouring origins overlap (origins every 14 days, horizon up to 30), so `n` counts
forecasts, not unique days.

## 8. Actual vs predicted

For readability, each date shows the forecast from the **most recent origin** before it (the shortest lead
time, at most 14 days). The chart zooms into one term; the table summarises the whole test period on the same basis.

In [ ]:
# @title 8. Choose a term to plot
TERM_TO_PLOT = "2025-T2"  # @param ["2024-T1", "2024-T2", "2024-T3", "2025-T1", "2025-T2", "2025-T3"]

test_preds = preds[(preds["split"] == "test") & preds["model"].isin(selected)]
latest = test_preds.sort_values("days_ahead").drop_duplicates(["model", "date"]).sort_values("date")
term_dates = set(school.loc[school["term"] == TERM_TO_PLOT, "date"])
view = latest[latest["date"].isin(term_dates)]
actual = view.drop_duplicates("date")[["date", "actual"]]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(actual["date"], actual["actual"], color=INK, linewidth=2, marker="o", markersize=4, label="Actual demand")
for name in selected:
    s = view[view["model"] == name]
    ax.plot(s["date"], s["yhat"], color=model_color(name), linewidth=2, label=pretty(name))
ax.set_title(f"Actual vs predicted demand, {TERM_TO_PLOT} (forecast from the most recent origin)", pad=30)
ax.set_ylabel("meals"); legend_above(ax, 4)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.tight_layout(); plt.show()

err = latest.assign(abs_error=(latest["yhat"] - latest["actual"]).abs())
display(err.groupby("model").agg(forecasts=("abs_error", "size"), MAE=("abs_error", "mean"),
                                 bias=("yhat", lambda s: float((s - err.loc[s.index, "actual"]).mean())))
           .rename(index=pretty).round(2))

## 9. Prediction intervals

Prophet's 80% and 95% intervals for the selected model on the test period, again using the most recent origin
for each date. **Empirical coverage** is the share of actual values that fell inside each interval.
Intervals assume the future behaves like the training data; they are not guaranteed bounds, and because the
data is synthetic, coverage here says nothing reliable about uncertainty for a real school.

In [ ]:
pname = evaluation["selected"]["prophet"]["name"]
pv = latest[latest["model"] == pname]
pv = pv[pv["date"].isin(term_dates)]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(pv["date"], pv["lower_95"], pv["upper_95"], color=COLORS["prophet"], alpha=0.12, linewidth=0, label="95% interval")
ax.fill_between(pv["date"], pv["lower_80"], pv["upper_80"], color=COLORS["prophet"], alpha=0.25, linewidth=0, label="80% interval")
ax.plot(pv["date"], pv["yhat"], color=COLORS["prophet"], linewidth=2, label="Prophet forecast")
outside = (pv["actual"] < pv["lower_80"]) | (pv["actual"] > pv["upper_80"])
ax.scatter(pv.loc[~outside, "date"], pv.loc[~outside, "actual"], s=22, color=INK, zorder=3, label="Actual (inside 80%)")
ax.scatter(pv.loc[outside, "date"], pv.loc[outside, "actual"], s=36, facecolors=SURFACE, edgecolors=INK,
           linewidths=1.5, zorder=3, label="Actual (outside 80%)")
ax.set_title(f"Prophet prediction intervals, {TERM_TO_PLOT}", pad=30)
ax.set_ylabel("meals"); legend_above(ax, 5)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.tight_layout(); plt.show()

cov = pd.concat([val_summary.assign(split="validation"), test_summary.assign(split="test")])
cov = cov[cov["model"] == pname][["split", "horizon_days", "n", "coverage_80", "coverage_95"]]
display(cov.assign(coverage_80=lambda d: (100 * d["coverage_80"]).round(1),
                   coverage_95=lambda d: (100 * d["coverage_95"]).round(1))
           .rename(columns={"coverage_80": "inside 80% interval (%)", "coverage_95": "inside 95% interval (%)"})
           .reset_index(drop=True))

The intervals are somewhat too narrow: fewer actual values fall inside them than the nominal 80% / 95%,
especially on the test period. Treat them as a rough indication of spread.

## 10. Configurable future forecast

This uses the **saved model** (`models/artifacts/`) through the same `ForecastService` the Flask API uses.
No retraining. Only school days from the configured calendar are returned. The saved model was trained on data
up to 2025-10-31; start dates must be after that and within 730 days of it.

In [ ]:
# @title 10. Forecast settings
START_DATE = "2026-10-05"  # @param {type:"date"}
HORIZON_DAYS = 30  # @param {type:"slider", min:1, max:60, step:1}
ENROLLMENT = 600  # @param {type:"integer"}
# Set ENROLLMENT to 0 to use the last enrollment in the training data.

service = ForecastService()
if not service.load():
    raise RuntimeError(service.load_error)
params = {"start_date": START_DATE, "horizon_days": HORIZON_DAYS}
if ENROLLMENT:
    params["enrollment"] = ENROLLMENT
try:
    start, horizon, enrollment = service.parse_request(params)
    result = service.forecast(start, horizon, enrollment)
except ForecastError as e:
    raise ValueError(f"Request rejected ({e.status_code}): {e}") from None

print(f"Model: {result['model_name']} (trained {result['trained_at']}, data to {result['training_data_end']})")
print(f"{result['start_date']} .. {result['end_date']}: {result['school_days_forecast']} school days forecast, "
      f"{result['non_school_days_excluded']} non-school days excluded")
for a in result["assumptions"]:
    print(" -", a)
print(result["data_notice"])
future_fc = pd.DataFrame(result["forecast"])
display(future_fc)

In [ ]:
if future_fc.empty:
    print("No school days in this window (holiday period) - nothing to plot.")
else:
    d = pd.to_datetime(future_fc["date"])
    fig, ax = plt.subplots(figsize=(12, 4.2))
    yerr = [future_fc["predicted_demand"] - future_fc["lower_bound"], future_fc["upper_bound"] - future_fc["predicted_demand"]]
    ax.errorbar(d, future_fc["predicted_demand"], yerr=yerr, fmt="o", color=COLORS["prophet"], markersize=6,
                ecolor=COLORS["prophet"], elinewidth=2, capsize=0, alpha=0.9,
                label=f"Predicted demand with {int(result['interval_width'] * 100)}% interval (school days only)")
    ax.set_title(f"Forecast of meal demand, {result['start_date']} to {result['end_date']}", pad=30)
    ax.set_ylabel("meals"); legend_above(ax, 1)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("Mon %d %b"))
    fig.autofmt_xdate()
    plt.tight_layout(); plt.show()

## 11. (Optional) Run the project's automated tests

Runs the same pytest suite as the local project (about 10 seconds).

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--color=no", "-p", "no:cacheprovider"],
                     capture_output=True, text=True).stdout[-1500:])

## Limitations

* The data is synthetic and represents one simulated Kenyan school; no model has been validated against real records.
* Every relationship in the data was written into the generator, so the models can at best rediscover those assumptions.
* Enrollment is held constant over the forecast horizon; weather and special events are not used.
* The future school calendar uses illustrative rules, not official Ministry of Education dates.
* The forecasting service is not yet integrated with the Node.js backend or React frontend.